In [1]:
import polars as pl
from pathlib import Path

DATA_DIR = Path("./csiro-biomass")

train = pl.read_csv(DATA_DIR / "train.csv")
test = pl.read_csv(DATA_DIR / "test.csv")
train

sample_id,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,target_name,target
str,str,str,str,str,f64,f64,str,f64
"""ID1011485656__Dry_Clover_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Clover_g""",0.0
"""ID1011485656__Dry_Dead_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Dead_g""",31.9984
"""ID1011485656__Dry_Green_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Green_g""",16.2751
"""ID1011485656__Dry_Total_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""Dry_Total_g""",48.2735
"""ID1011485656__GDM_g""","""train/ID1011485656.jpg""","""2015/9/4""","""Tas""","""Ryegrass_Clover""",0.62,4.6667,"""GDM_g""",16.275
…,…,…,…,…,…,…,…,…
"""ID983582017__Dry_Clover_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Clover_g""",0.0
"""ID983582017__Dry_Dead_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Dead_g""",0.0
"""ID983582017__Dry_Green_g""","""train/ID983582017.jpg""","""2015/9/1""","""WA""","""Ryegrass""",0.64,9.0,"""Dry_Green_g""",40.94


In [2]:
train_cleaned = train.pivot(on='target_name', values='target', index=['image_path'])
test_modified = test.with_columns(pl.lit(0).alias('target'))
test_cleaned = test_modified.pivot(on='target_name', values='target', index=['image_path'])
train_cleaned.head()

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,f64,f64,f64,f64,f64
"""train/ID1011485656.jpg""",0.0,31.9984,16.2751,48.2735,16.275
"""train/ID1012260530.jpg""",0.0,0.0,7.6,7.6,7.6
"""train/ID1025234388.jpg""",6.05,0.0,0.0,6.05,6.05
"""train/ID1028611175.jpg""",0.0,30.9703,24.2376,55.2079,24.2376
"""train/ID1035947949.jpg""",0.4343,23.2239,10.5261,34.1844,10.9605


In [3]:
test_cleaned.head(10)

image_path,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
str,i32,i32,i32,i32,i32
"""test/ID1001187975.jpg""",0,0,0,0,0


In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split

class BiomassDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df
        self.img_dir = img_dir
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.row(idx)
        img_path = row[0]
        image = Image.open(self.img_dir / img_path)
        if self.transform:
            image = self.transform(image)
        labels = torch.tensor(row[1:], dtype=torch.float32)
        return image, labels

train_df, val_df = train_test_split(train_cleaned, test_size=0.2, random_state=42)
train_dataset = BiomassDataset(train_df, DATA_DIR)
val_dataset = BiomassDataset(val_df, DATA_DIR)
test_dataset = BiomassDataset(test_cleaned, DATA_DIR)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

for images, labels in train_loader:
    print(images.shape, labels.shape)
    break   

torch.Size([32, 3, 224, 224]) torch.Size([32, 5])


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

# Detect CUDA device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

class BiomassModel(nn.Module):
    def __init__(self, num_targets):
        super(BiomassModel, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 56 * 56, 128),
            nn.ReLU(),
            nn.Linear(128, num_targets),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x
    
num_targets = train_cleaned.shape[1] - 1
model = BiomassModel(num_targets)
model = model.to(device)  # Move model to GPU/CPU

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        # Move data to device
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    epoch_loss = running_loss / len(train_dataset)
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}')


Using device: cuda
Epoch 1/10, Loss: 716.3591
Epoch 2/10, Loss: 532.6292
Epoch 3/10, Loss: 488.6919
Epoch 4/10, Loss: 459.4909
Epoch 5/10, Loss: 447.8879
Epoch 6/10, Loss: 419.3893
Epoch 7/10, Loss: 408.0823
Epoch 8/10, Loss: 395.7402
Epoch 9/10, Loss: 379.2429
Epoch 10/10, Loss: 363.2480


In [7]:
outputs = []
model.eval()
with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        preds = model(images)
        preds = preds.cpu()
        outputs.append(preds)

outputs = torch.cat(outputs, dim=0).numpy()
outputs

array([[ 4.9192123, 13.641708 , 31.347275 , 44.59202  , 35.332108 ]],
      dtype=float32)

In [8]:
sample_submission = pl.read_csv(DATA_DIR / "sample_submission.csv")
# outputs is already a concatenated tensor from the previous cell
sample_submission = sample_submission.with_columns(pl.Series("target", outputs.flatten()))
sample_submission

sample_id,target
str,f32
"""ID1001187975__Dry_Clover_g""",4.919212
"""ID1001187975__Dry_Dead_g""",13.641708
"""ID1001187975__Dry_Green_g""",31.347275
"""ID1001187975__Dry_Total_g""",44.592018
"""ID1001187975__GDM_g""",35.332108


In [16]:
sample_submission.write_csv("./submissions/submission_1.csv")